# Anomalías industriales en sensores: Preparación y controles de datos

Ejecuta la etapa anterior primero. Las cifras y conclusiones siguientes se calculan con los datos de esta ejecución; no edites manualmente sus valores. Consulta [metodología](../docs/metodologia.md) para interpretar resultados.

In [1]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("industrial_anomaly_detection")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre este notebook desde su proyecto independiente.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from industrial_anomaly_detection.alerts import configurar_notebook, mostrar_alertas
from IPython.display import display, Markdown, Image
configurar_notebook()


In [2]:
from industrial_anomaly_detection.analysis import run_stage
import pandas as pd
metrics, comment = run_stage(PROJECT_ROOT, "preparacion")
display(Markdown(comment))
scalars = {k: v for k, v in metrics.items() if not isinstance(v, (dict, list))}
if scalars:
    display(pd.DataFrame(list(scalars.items()), columns=["Indicador", "Valor"]))
if "candidatos" in metrics:
    display(pd.DataFrame(metrics["candidatos"]))
if "baseline" in metrics and "modelo" in metrics:
    comparison = {key: {k: v for k, v in metrics[key].items() if not isinstance(v, (dict, list))} for key in ["baseline", "modelo"]}
    display(pd.DataFrame(comparison).T)
if "clasificacion" in metrics.get("modelo", {}):
    display(pd.DataFrame(metrics["modelo"]["clasificacion"]).T)
for image_name in ["eda.png", "evaluacion.png"]:
    image_path = PROJECT_ROOT / "reports/preparacion" / image_name
    if image_path.exists():
        display(Image(filename=str(image_path)))


# Anomalías industriales en sensores: preparacion

AI4I es un **dataset sintético**, no registros de una planta. Se separan filas 60/20/20 de forma estratificada. Se retiran combinaciones de sensores repetidas para evitar solapamientos. Los IDs, tipo de producto y todas las etiquetas de modos de fallo se excluyen como entradas. No hay fecha que permita afirmar validación temporal.


,Indicador,Valor
0,fuente_sintetica,True
1,sha256,f601f14294bcf190f9d720676b7f0aea46a26cde9ab8eb...
2,filas_utilizables,10000


In [3]:
mostrar_alertas(PROJECT_ROOT / "reports/alertas/01_preparacion_datos.json")
from industrial_anomaly_detection.informe import generar_informe
generar_informe(PROJECT_ROOT)
display(Markdown("[Abrir informe integral](../reports/informe_final/informe_final.md)"))


[Abrir informe integral](../reports/informe_final/informe_final.md)